In [1]:
import pandas as pd
import numpy as np

In [2]:
import os


In [5]:
import re

def process_chemical_formula(formula):
    # Regular expression to match molecules in parentheses and their counts
    molecule_pattern = re.compile(r'\(([A-Za-z0-9]+)\)(\d*)')
    # Regular expression to match atoms and their counts
    atom_pattern = re.compile(r'([A-Z][a-z]*)(\d*)')

    # Dictionary to store the count of each atom
    atom_counts = {}

    # Function to update atom counts
    def update_atom_counts(atom, count):
        if atom not in atom_counts:
            atom_counts[atom] = count
        else:
            atom_counts[atom] += count

    # Extracting molecules in parentheses and their counts
    for molecule_match in molecule_pattern.finditer(formula):
        molecule, mol_count = molecule_match.groups()
        mol_count = int(mol_count) if mol_count else 1

        # Extract atoms from the molecule
        for atom_match in atom_pattern.finditer(molecule):
            atom, count = atom_match.groups()
            count = int(count) if count else 1
            update_atom_counts(atom, count * mol_count)

        # Remove the molecule from the formula
        formula = formula.replace(molecule_match.group(), '', 1)

    # Extracting remaining atoms and their counts
    for match in atom_pattern.finditer(formula):
        atom, count = match.groups()
        # If count is empty, set it to 1
        count = int(count) if count else 1

        # Ignore charge information
        if atom.endswith('+') or atom.endswith('-'):
            continue

        update_atom_counts(atom, count)

    # Convert the counts to a list format
    atom_list = []
    for atom, count in atom_counts.items():
        atom_list.extend([atom] * count)

    return atom_list

# Test the function with the given formula
formula = "Mn^2+^_13_Al_4_Sb^5+^_2_O_20_(SiO4_)_2_"
print(process_chemical_formula(formula))


['Mn', 'Al', 'Sb', 'O', 'O', 'O', 'O', 'O', 'Si']


In [180]:
import os
import re

def extract_chemistry_strings(directory):
    # Regular expression to match the desired pattern
    pattern = r'##IDEAL CHEMISTRY=(.+)'

    # Dictionary to store the results: {filename: [list_of_extracted_strings]}
    extracted_strings = {}

    # Iterate over each file in the directory
    for filename in os.listdir(directory):
        file_path = os.path.join(directory, filename)

        # Check if it's a file and not a directory
        if os.path.isfile(file_path):
            with open(file_path, 'r') as file:
                # Read each line in the file
                for line in file:
                    # Search for the pattern in the line
                    match = re.search(pattern, line)
                    if match:
                        # Extract the string after the equals sign
                        chemistry_string = match.group(1)

                        # Add the extracted string to the dictionary
                        if filename not in extracted_strings:
                            extracted_strings[filename] = []
                        extracted_strings[filename].append(chemistry_string)

    return extracted_strings

# Example usage
#/home/gridsan/tmackey/cdvae/scripts/1-06-2024_augmentation/XRDs/XY_RAW
directory_path = '/home/gridsan/tmackey/cdvae/scripts/1-06-2024_augmentation/XRDs/XY_RAW/'
result = extract_chemistry_strings(directory_path)
print(result)


{'Vanadinite__R050376-1__Powder__Xray_Data_XY_RAW__1604.txt': ['Pb^2+^_5_(V^5+^O_4_)_3_Cl'], 'Bariopharmacosiderite__R060571-9__Powder__Xray_Data_XY_RAW__11649.txt': ['Ba_0.5_Fe^3+^_4_(As^5+^O_4_)_3_(OH)_4_&#183;5H_2_O'], 'Aragonite__R060195-1__Powder__Xray_Data_XY_RAW__2378.txt': ['CaCO_3_'], 'Carraraite__R130410-9__Powder__Xray_Data_XY_RAW__12073.txt': ['Ca_3_Ge(S^6+^O_4_)(CO_3_)(OH)_6_&#183;12H_2_O'], 'Grossular__R060499-1__Powder__Xray_Data_XY_RAW__4887.txt': ['Ca_3_Al_2_(SiO_4_)_3_'], 'Titanite__R100025-9__Powder__Xray_Data_XY_RAW__10251.txt': ['CaTi^4+^SiO_5_'], 'Scawtite__R060340-9__Powder__Xray_Data_XY_RAW__12399.txt': ['Ca_7_(Si_3_O_9_)_2_(CO_3_)&#183;2H_2_O'], 'Rutherfordine__R110154-9__Powder__Xray_Data_XY_RAW__10979.txt': ['(U^6+^O_2_)CO_3_'], 'Ulexite__R050491-1__Powder__Xray_Data_XY_RAW__1798.txt': ['NaCaB_5_O_6_(OH)_6_&#183;5H_2_O'], 'Ussingite__R070562-1__Powder__Xray_Data_XY_RAW__8775.txt': ['Na_2_AlSi_3_O_8_(OH)'], 'Sekaninaite__R061053-1__Powder__Xray_Data_XY_RAW__74

In [235]:
index = 10
print(result[list(result.keys())[index]][0])

Fe^2+^_2_Al_3_(AlSi_5_)O_18_


In [238]:
list_of_results = []
for index in range(len(result)):

    try:
        formula = result[list(result.keys())[index]][0]
        original_formula = formula
        #remove anything between ^ and ^ from the formula
        formula = re.sub(r'\^.*?\^', '', formula)
        pattern = r'\(.*?\)_?\d*_'

        def atom_counts(molecule):
            # Regular expression to find elements and their counts
            atom_pattern = re.compile(r'([A-Z][a-z]*)(\d*)')
            counts = {}

            for atom, count in atom_pattern.findall(molecule):
                # If no count is specified, default to 1
                count = int(count) if count else 1

                if atom in counts:
                    counts[atom] += count
                else:
                    counts[atom] = count

            return counts

        list_of_molecule_dicts = []
        molecule_match = re.finditer(pattern, formula)
        for one_match in molecule_match:

            matched_text = one_match.group().replace("(", "")
            matched_text = matched_text.replace(")", "")

            matched_text = matched_text.split("__")

            molecule = matched_text[0].replace("_", "")
            count = int(matched_text[1].replace("_", ""))

            print(matched_text)

            one_dict = {}
            
            counts = atom_counts(molecule)

            #loop over all atoms in the molecule
            for atom, atom_count in counts.items():
                #update the atom count
                atom_count = atom_count * count

                one_dict[atom] = atom_count
                
            list_of_molecule_dicts.append(one_dict)

            #remove the match from the formula
            formula = formula.replace(one_match.group(), '')

        total_dict = {}

        for atom_dict in list_of_molecule_dicts:
            for key, value in atom_dict.items():
                if key in total_dict:
                    total_dict[key] += value
                else:
                    total_dict[key] = value

        list_atoms = formula.replace("_", "")

        not_molecule_dict = atom_counts(list_atoms)

        for key, value in not_molecule_dict.items():
            if key in total_dict:
                total_dict[key] += value
            else:
                total_dict[key] = value


        print(total_dict)
        print(original_formula)
        print(formula)

        list_of_results.append(total_dict)
    except:
        continue



['VO_4', '3_']
{'V': 3, 'O': 12, 'Pb': 5, 'Cl': 1}
Pb^2+^_5_(V^5+^O_4_)_3_Cl
Pb_5_Cl
['AsO_4', '3_']
{'Ca': 1, 'C': 1, 'O': 3}
CaCO_3_
CaCO_3_
['SiO_4', '3_']
{'Si': 3, 'O': 12, 'Ca': 3, 'Al': 2}
Ca_3_Al_2_(SiO_4_)_3_
Ca_3_Al_2_
{'Ca': 1, 'Ti': 1, 'Si': 1, 'O': 5}
CaTi^4+^SiO_5_
CaTiSiO_5_
['Si_3_O_9', '2_']
{'Si': 6, 'O': 22, 'Ca': 7, 'C': 1, 'H': 2}
Ca_7_(Si_3_O_9_)_2_(CO_3_)&#183;2H_2_O
Ca_7_(CO_3_)&#183;2H_2_O
{'U': 1, 'O': 5, 'C': 1}
(U^6+^O_2_)CO_3_
(UO_2_)CO_3_
{'Na': 2, 'Al': 1, 'Si': 3, 'O': 9, 'H': 1}
Na_2_AlSi_3_O_8_(OH)
Na_2_AlSi_3_O_8_(OH)
{'Fe': 2, 'Al': 4, 'Si': 5, 'O': 18}
Fe^2+^_2_Al_3_(AlSi_5_)O_18_
Fe_2_Al_3_(AlSi_5_)O_18_
['SiO_4', '3_']
{'Si': 3, 'O': 12, 'Ca': 3, 'Fe': 2}
Ca_3_Fe^3+^_2_(SiO_4_)_3_
Ca_3_Fe_2_
['SO_4', '6_']
['SiO_4', '3_']
{'Si': 3, 'O': 12, 'Ca': 3, 'V': 2}
Ca_3_V^3+^_2_(SiO_4_)_3_
Ca_3_V_2_
{'Mg': 1, 'B': 1, 'O': 3, 'H': 1}
MgBO_2_(OH)
MgBO_2_(OH)
{'Ag': 1, 'Mn': 1, 'Pb': 3, 'Sb': 5, 'S': 12}
Ag^1+^Mn^2+^Pb^2+^_3_Sb^3+^_5_S^2-^_12_
AgMnPb_3_Sb_5_

In [239]:
len(list_of_results)

2055

In [243]:
list_of_results

[{'V': 3, 'O': 12, 'Pb': 5, 'Cl': 1},
 {'Ca': 1, 'C': 1, 'O': 3},
 {'Si': 3, 'O': 12, 'Ca': 3, 'Al': 2},
 {'Ca': 1, 'Ti': 1, 'Si': 1, 'O': 5},
 {'Si': 6, 'O': 22, 'Ca': 7, 'C': 1, 'H': 2},
 {'U': 1, 'O': 5, 'C': 1},
 {'Na': 2, 'Al': 1, 'Si': 3, 'O': 9, 'H': 1},
 {'Fe': 2, 'Al': 4, 'Si': 5, 'O': 18},
 {'Si': 3, 'O': 12, 'Ca': 3, 'Fe': 2},
 {'Si': 3, 'O': 12, 'Ca': 3, 'V': 2},
 {'Mg': 1, 'B': 1, 'O': 3, 'H': 1},
 {'Ag': 1, 'Mn': 1, 'Pb': 3, 'Sb': 5, 'S': 12},
 {'Ca': 4, 'Si': 28, 'Al': 8, 'O': 73, 'H': 2},
 {'Sb': 2, 'S': 3},
 {'Li': 1, 'Al': 1, 'Si': 1, 'O': 4},
 {'As': 2, 'O': 9, 'Co': 3, 'H': 2},
 {'Cu': 1, 'Co': 2, 'S': 4},
 {'Pb': 1, 'Sn': 1, 'S': 2},
 {'Co': 1, 'Fe': 1, 'Ni': 1, 'As': 2},
 {'Mn': 1, 'Ta': 2, 'O': 6},
 {'Ca': 2, 'Pb': 1, 'Si': 3, 'O': 9},
 {'K': 1, 'Na': 2, 'Fe': 2, 'Li': 3, 'Si': 12, 'O': 30},
 {'Ca': 1, 'Sr': 1, 'Mn': 2, 'Al': 1, 'Si': 3, 'O': 13, 'H': 1},
 {'Al': 2, 'O': 5, 'Si': 1},
 {'Ca': 1, 'Zn': 1, 'Si': 1, 'O': 5, 'H': 2},
 {'Si': 4, 'O': 15, 'Ca': 6, 'Na':

In [244]:
#only take dictionaries where the sum of the values is less than 0r equal to 20
list_of_results_20 = []
for one_dict in list_of_results:
    if sum(one_dict.values()) <= 20:
        list_of_results_20.append(one_dict)

In [245]:
len(list_of_results_20)

1519

In [247]:
(list_of_results_20)

[{'Ca': 1, 'C': 1, 'O': 3},
 {'Si': 3, 'O': 12, 'Ca': 3, 'Al': 2},
 {'Ca': 1, 'Ti': 1, 'Si': 1, 'O': 5},
 {'U': 1, 'O': 5, 'C': 1},
 {'Na': 2, 'Al': 1, 'Si': 3, 'O': 9, 'H': 1},
 {'Si': 3, 'O': 12, 'Ca': 3, 'Fe': 2},
 {'Si': 3, 'O': 12, 'Ca': 3, 'V': 2},
 {'Mg': 1, 'B': 1, 'O': 3, 'H': 1},
 {'Sb': 2, 'S': 3},
 {'Li': 1, 'Al': 1, 'Si': 1, 'O': 4},
 {'As': 2, 'O': 9, 'Co': 3, 'H': 2},
 {'Cu': 1, 'Co': 2, 'S': 4},
 {'Pb': 1, 'Sn': 1, 'S': 2},
 {'Co': 1, 'Fe': 1, 'Ni': 1, 'As': 2},
 {'Mn': 1, 'Ta': 2, 'O': 6},
 {'Ca': 2, 'Pb': 1, 'Si': 3, 'O': 9},
 {'Al': 2, 'O': 5, 'Si': 1},
 {'Ca': 1, 'Zn': 1, 'Si': 1, 'O': 5, 'H': 2},
 {'Cu': 1, 'Se': 1, 'O': 4, 'H': 2},
 {'N': 2, 'H': 8, 'S': 1, 'O': 4},
 {'Mg': 2, 'Si': 1, 'O': 4},
 {'Mn': 1, 'C': 1, 'O': 3},
 {'Se': 3, 'O': 10, 'Al': 2, 'H': 2},
 {'Mg': 1, 'Al': 2, 'O': 4},
 {'Mn': 5, 'As': 3, 'O': 10, 'H': 1, 'Cl': 1},
 {'Si': 3, 'O': 12, 'Ca': 3, 'Fe': 2},
 {'Ca': 1, 'Cu': 1, 'As': 1, 'O': 5, 'H': 1},
 {'Pb': 2, 'Al': 1, 'Si': 4, 'O': 12, 'H': 1},


In [74]:
train_df = pd.read_csv("/home/gridsan/tmackey/cdvae/data/mp_20/train.csv")

In [75]:
train_df

,Unnamed: 0,material_id,formation_energy_per_atom,band_gap,pretty_formula,e_above_hull,elements,cif,spacegroup.number,xrd,xrd_peak_locations,xrd_peak_intensities,atomic_numbers,disc_sim_xrd
0,37228,mp-1221227,-1.637460,0.2133,Na3MnCoNiO6,0.043001,"['Co', 'Mn', 'Na', 'Ni', 'O']",# generated using pymatgen\ndata_Na3MnCoNiO6\n...,8,DiffractionPattern\n$2\Theta$: [11.87011609 16...,"[11.87011609469122, 16.52069689929858, 17.1482...","[0.464671693703146, 94.05248185142216, 0.72569...","[11, 11, 11, 25, 27, 28, 8, 8, 8, 8, 8, 8]",[0.00000000e+00 0.00000000e+00 0.00000000e+00 ...
1,19480,mp-974729,-0.314759,0.0000,Nd(Al2Cu)4,0.000000,"['Al', 'Cu', 'Nd']",# generated using pymatgen\ndata_Nd(Al2Cu)4\n_...,139,DiffractionPattern\n$2\Theta$: [14.07069605 19...,"[14.070696049844747, 19.775602977923736, 19.94...","[29.452478536673507, 29.524371989407847, 12.11...","[60, 13, 13, 13, 13, 13, 13, 13, 13, 29, 29, 2...",[0.00000000e+00 0.00000000e+00 0.00000000e+00 ...
2,29624,mp-1185360,-0.193761,0.0000,LiMnIr2,0.018075,"['Ir', 'Li', 'Mn']",# generated using pymatgen\ndata_LiMnIr2\n_sym...,225,DiffractionPattern\n$2\Theta$: [26.20871065 30...,"[26.2087106549449, 30.353774524330394, 43.4609...","[3.6250325610399834, 68.2501747275368, 100.0, ...","[3, 25, 77, 77]",[ 0. 0. 0. 0. ...
3,38633,mp-1188861,-0.584694,3.8556,LiCSN,0.048847,"['C', 'Li', 'N', 'S']",# generated using pymatgen\ndata_LiCSN\n_symme...,62,DiffractionPattern\n$2\Theta$: [14.35172042 18...,"[14.351720419152775, 18.084725325124687, 21.99...","[9.98753987807856, 12.13654435473029, 1.800790...","[3, 3, 3, 3, 6, 6, 6, 6, 16, 16, 16, 16, 7, 7,...",[0.00000000e+00 0.00000000e+00 0.00000000e+00 ...
4,10889,mp-677272,-2.474759,0.4707,La2EuS4,0.000000,"['Eu', 'La', 'S']",# generated using pymatgen\ndata_La2EuS4\n_sym...,122,DiffractionPattern\n$2\Theta$: [14.32156225 20...,"[14.32156225226108, 20.229215345610065, 24.873...","[0.4719172724252014, 0.09181220173027804, 100....","[57, 57, 57, 57, 63, 63, 16, 16, 16, 16, 16, 1...",[0.00000000e+00 0.00000000e+00 0.00000000e+00 ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27131,37856,mp-568116,-0.988502,3.7614,Lu2(CN2)3,0.045185,"['C', 'Lu', 'N']",# generated using pymatgen\ndata_Lu2(CN2)3\n_s...,155,DiffractionPattern\n$2\Theta$: [17.45526854 17...,"[17.45526854002256, 17.729775284134913, 20.261...","[69.34392053889913, 51.93665208875416, 51.6455...","[71, 71, 6, 6, 6, 7, 7, 7, 7, 7, 7]",[0.00000000e+00 0.00000000e+00 0.00000000e+00 ...
27132,11955,mp-865529,-0.640955,0.0000,Ti2NiIr,0.000000,"['Ti', 'Ni', 'Ir']",# generated using pymatgen\ndata_Ti2NiIr\n_sym...,225,DiffractionPattern\n$2\Theta$: [25.17533412 29...,"[25.175334115198098, 29.1500541146744, 41.6951...","[31.130319327843168, 27.041046017803335, 100.0...","[22, 22, 28, 77]",[ 0. 0. 0. 0. ...
27133,26119,mp-1189241,-0.756019,0.0000,GdAs2Au,0.010305,"['As', 'Au', 'Gd']",# generated using pymatgen\ndata_GdAs2Au\n_sym...,64,DiffractionPattern\n$2\Theta$: [ 8.62499477 17...,"[8.624994768713732, 17.29934209744178, 22.3945...","[94.58736579948653, 5.246648064082296, 0.08933...","[64, 64, 64, 64, 33, 33, 33, 33, 33, 33, 33, 3...",[0.00000000e+00 0.00000000e+00 0.00000000e+00 ...
27134,30556,mp-1104538,-0.104870,0.0000,Tm(FeSn)6,0.021206,"['Fe', 'Sn', 'Tm']",# generated using pymatgen\ndata_Tm(FeSn)6\n_s...,191,DiffractionPattern\n$2\Theta$: [ 9.9202585 18...,"[9.920258495206648, 18.934413061155865, 19.915...","[2.4962176053163656, 1.3457049100913736, 5.814...","[69, 26, 26, 26, 26, 26, 26, 50, 50, 50, 50, 5...",[0.00000000e+00 0.00000000e+00 0.00000000e+00 ...


In [17]:
for molecule_match in molecule_pattern.finditer(formula):
    print(molecule_match.groups())

('SiO4_', '')


In [9]:
for atom_match in atom_pattern.finditer(formula):
    print(atom_match.groups())

('Mn', '')
('Al', '')
('Sb', '')
('O', '')
('Si', '')
('O', '4')


In [ ]:

# Dictionary to store the count of each atom
atom_counts = {}

# Function to update atom counts
def update_atom_counts(atom, count):
    if atom not in atom_counts:
        atom_counts[atom] = count
    else:
        atom_counts[atom] += count

# Extracting molecules in parentheses and their counts
for molecule_match in molecule_pattern.finditer(formula):
    molecule, mol_count = molecule_match.groups()
    mol_count = int(mol_count) if mol_count else 1

    # Extract atoms from the molecule
    for atom_match in atom_pattern.finditer(molecule):
        atom, count = atom_match.groups()
        count = int(count) if count else 1
        update_atom_counts(atom, count * mol_count)

    # Remove the molecule from the formula
    formula = formula.replace(molecule_match.group(), '', 1)

# Extracting remaining atoms and their counts
for match in atom_pattern.finditer(formula):
    atom, count = match.groups()
    # If count is empty, set it to 1
    count = int(count) if count else 1

    # Ignore charge information
    if atom.endswith('+') or atom.endswith('-'):
        continue

    update_atom_counts(atom, count)


In [ ]:

    # Convert the counts to a list format
    atom_list = []
    for atom, count in atom_counts.items():
        atom_list.extend([atom] * count)

    return atom_list